# Histopathology image classification — notebook walkthrough

**Research-only, NOT a diagnostic system.**

This notebook makes the project's workflow visible: data provenance → image and label checks → patient-group separation → baseline/CNN experiments → patient-level evaluation.

**Data status:** no real pathology image dataset is packaged or validated in the repository. The existing synthetic color-noise fixtures are only software smoke tests, **not pathology**. This notebook will not generate artificial training images or report diagnostic scores. Supply properly licensed real images, verified patient identities, and the provenance manifest first.

This walkthrough is a companion created after the modular source; it does not purport to be an original development record.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "README.md").exists() and (ROOT.parent / "README.md").exists():
    ROOT = ROOT.parent
if not (ROOT / "README.md").exists():
    raise RuntimeError("Run the notebook from the project root or notebooks/ directory")
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)

## 1. Inspect the patient-level data contract
Patient-separated evaluation reduces leakage from images belonging to the same patient. It does not fix confounding by scanner, hospital or stain.

In [ ]:
print((ROOT/"docs"/"DATA.md").read_text(encoding="utf-8")[:9000])
print("\nEVALUATION PROTOCOL\n")
print((ROOT/"docs"/"EVALUATION.md").read_text(encoding="utf-8")[:4500])

## 2. Validate genuine image manifest and review class distribution
Required columns: `image_id,patient_id,path,label`. Labels: 0 benign, 1 malignant. Paths must point to genuine, authorized local images.

In [ ]:
import pandas as pd
from pathlib import Path
manifest=ROOT/"data"/"manifest.csv"
provenance=ROOT/"data"/"provenance.json"
if not manifest.is_file() or not provenance.is_file():
    raise FileNotFoundError("Provide real pathology data/manifest.csv and data/provenance.json first. No synthetic fallback.")
df=pd.read_csv(manifest)
required={"image_id","patient_id","path","label"}
assert required.issubset(df.columns),sorted(required-set(df.columns))
print("Image records:",len(df),"Patients:",df.patient_id.nunique())
display(df.groupby("label").agg(images=("image_id","size"),patients=("patient_id","nunique")))
assert df.groupby("patient_id").label.nunique().eq(1).all(), "Conflicting patient labels"
assert set(df.label.unique())=={0,1}, "Expected binary labels"

## 3. Inspect example images (only after provenance and permissions are verified)
This cell limits display to a small, locally authorized sample.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
sample=df.sample(n=min(4,len(df)),random_state=42)
fig,axes=plt.subplots(1,len(sample),figsize=(3*len(sample),3))
axes=[axes] if len(sample)==1 else axes
for ax,(_,item) in zip(axes,sample.iterrows()):
    local=(ROOT/"data"/str(item["path"])).resolve()
    assert local.is_relative_to((ROOT/"data").resolve()),"Refusing path traversal"
    with Image.open(local) as im: ax.imshow(im.convert("RGB"))
    ax.set_title(f'Label {item["label"]}');ax.axis("off")
plt.tight_layout();plt.show()

## 4. Real-data experiment commands
Use the same manifest and group split for both models. From the project root:

In [ ]:
print('python -m histopathology validate --manifest data/manifest.csv --data-root data --provenance data/provenance.json --config configs/default.toml')
print('python -m histopathology run --model baseline --manifest data/manifest.csv --data-root data --provenance data/provenance.json --config configs/default.toml --output runs/real-baseline')
print('python -m histopathology run --model cnn --manifest data/manifest.csv --data-root data --provenance data/provenance.json --config configs/default.toml --output runs/real-cnn')

## 5. How notebook stages map to source code
- `data.py`: manifest, image checks, verified patient identities and splits.
- `features.py`: image processing and baseline color features.
- `models.py`: CPU CNN and training logic.
- `evaluation.py`: patient-level predictions, uncertainty and failures.
- `experiment.py`: experiment orchestration and run artifacts.
- `cli.py`: reproducible commands; `configs/default.toml`: explicit settings.

**Clinical limitation:** even successfully trained models require external cohort and prospective validation before any medical claim. No such claims are made here.